In [ ]:
import pandas as pd
import numpy as np
import re

dff1 = pd.read_csv('../data/cppt_ranap_jan maret 2024.csv')
dff2 = pd.read_csv('../data/cppt_ranap_apr juni 2024.csv')
dff3 = pd.read_csv('../data/cppt_ranap_juli 2024.csv')

In [ ]:
df = pd.concat([dff1, dff2, dff3], ignore_index=True)
df.info()

In [ ]:
def is_high_index_diagnosa(col):
    match = re.match(r"^result\[(\d+)\]\.", col)
    return match and int(match.group(1)) > 0 

df = df[[col for col in df.columns if not is_high_index_diagnosa(col)]]

df.info()

In [ ]:
patterns = [
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.diagnosaa$",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]$",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.jenisDiagnosa$",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.isCopy$",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.jenisDiagnosa.value$",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.isLoadBtnDiagnosaDokter$",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.diagnosaa.value$",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.type$",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.norecDiagnosa$",
    r"^user_input.",
    r"^profile.",
    r"^dpjpUtama",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.0",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.1",
    r"^result\[(\d+)\]\.emrpasienfk",
    r"^result\[(\d+)\]\.diagnosaDokter\[\d+\]\.no",
    r"^result\[(\d+)\]\.dokterDPJP",
    r"^result\[(\d+)\]\._id",
    r"skor",
    r"riwayatImunisasi",
]

for pattern in patterns:
    if not pattern:
        break
    df = df[[col for col in df.columns if not re.match(pattern, col)]]

df.info()

In [ ]:
def matching_function(pattern, col):
    match = re.match(pattern, col)
    return match and int(match.group(2))>=3

patterns2 = [
    r"^result\[(\d+)\]\.diagnosaDokter\[(\d+)\]\.",
    r"^result\[(\d+)\]\.diagnosaDokter(\d+)\[",
    r"^result\[(\d+)\]\.diagnosaDokter\[(\d+)\]\.keterangan$",
    r"^result\[(\d+)\]\.diagnosaDokter\[(\d+)\]\.jenisDiagnosa.label$",
]

for pattern in patterns2:
    if not pattern: break
    df = df[[col for col in df.columns if not matching_function(pattern, col)]]

df.info()


In [ ]:
# remove other unwanted cols
unwanted_cols = ["no", "uuid", "flag", "intruksiPPA", "kdprofile", "statusenabled", "diagnosaDokter[2].0.jenisDiagnosa.value", "diagnosaDokter[2].0.jenisDiagnosa.label", "diagnosaDokter[2].0.keterangan", "diagnosaDokter[2].0.diagnosaa.label", "diagnosaDokter[2].0.diagnosaa.value", "diagnosaDokter[2].0.type", "diagnosaDokter[2].1.jenisDiagnosa.label", "diagnosaDokter[2].1.jenisDiagnosa.value", "diagnosaDokter[2].1.keterangan", "diagnosaDokter[2].1.type", "keteranganVerifikasiDPJP", "tenagaMedis", "tglVerifikasi", "tgl", "DGizi","created_at","updated_at","update_count","update_before"]

def removeby (col, unwanted):
    pattern = r"^result\[(\d+)\]\." + unwanted
    match = re.match(pattern, col)
    return match

for unwanted in unwanted_cols:
    df = df[[col for col in df.columns if not removeby(col, unwanted)]]

df.info()

In [ ]:
# remove other unwanted cols
unwanted_cols = ["nocm","catatan_terbaru", "nocmfk","namapasien","suku","objectjeniskelaminfk","noidentitas","nobpjs","noasuransilain","alamatlengkap","kodepos","notelepon","nohp","namaayah","namaibu","email","agama","pendidikan","pekerjaan","isfoto","filename","isFilterProdukLab","enabledEMRSimrsLama","isclosing","objectagamafk"]

def removeby (col, unwanted):
    pattern = r"^pasien." + unwanted
    match = re.match(pattern, col)
    return match

for unwanted in unwanted_cols:
    df = df[[col for col in df.columns if not removeby(col, unwanted)]]

df.info()

In [ ]:
# remove other unwanted cols
unwanted_cols = ['norec_apd',"emrpasienfk",'norec_pd','noregistrasi','tglregistrasi','namarekanan','objectruanganfk','tglpulang','objectdepartemenfk','objectpegawaifk','asalrujukan','dokter']

def removeby (col, unwanted):
    pattern = r"^registrasi." + unwanted
    match = re.match(pattern, col)
    return match

for unwanted in unwanted_cols:
    df = df[[col for col in df.columns if not removeby(col, unwanted)]]

df.info()

In [ ]:
# remove other unwanted cols
unwanted_cols = ['dokterRawatBersama','id','created_at','updated_at','statusenabled','noemr']

df = df[[col for col in df.columns if col not in unwanted_cols]]

df.info()

In [ ]:
file_path = 'buang_col.txt'
columns_to_drop = []
with open(file_path, 'r') as file:
    for line in file:
        columns_to_drop.append(line.strip()) 

df.drop(columns=columns_to_drop, errors='ignore', inplace=True)

df.info()

In [ ]:
# remove the literal text  result[0].  from the start of every column name
df.columns = df.columns.str.replace(r'^result\[0\]\.', '', regex=True)
df.info()

In [ ]:
df2 = df.copy()
df2 = df2.dropna(subset=['diagnosaDokter[0].keterangan'])
df2 = df2[~df2['diagnosaDokter[0].keterangan'].str.contains('riw', case=False, na=False)]
df2 = df2[~df2['diagnosaDokter[0].keterangan'].str.contains('post', case=False, na=False)]
df2 = df2[~df2['diagnosaDokter[0].keterangan'].str.contains('pasca', case=False, na=False)]

df2.info()

In [ ]:
def replace_func(toreplace, replacedwith):
    df2.loc[df2['diagnosaDokter[0].keterangan'].str.contains(toreplace, case=False), 'grouped_diagnosa'] = replacedwith

In [ ]:
map1 = {
    'STEMI': 'ST-Elevation Myocardial Infarction',
    'angina': 'Angina Pectoris',
    'pectoris': 'Angina Pectoris',
    'aps': 'Angina Pectoris',
    'uap': 'Angina Pectoris',
    'ASHD':'Atherosclerosis Heart Disease',
    'AKI': 'Acute Kidney Disease',
    'CKD':'Chronic Kidney Disease',
    'CHF':'Chronic Heart Failure',
    'ACD': 'Coronary Artery Disease',
    'ACS': 'Acute Conorary Syndrome',
    'CAD': 'Coronary Artery Disease',
    'DM': 'Diabetes Mellitus',
    'ADHF': 'Acute Decompensated Heart Failure',
    'Chest pain': 'Chest Pain',
    'Palpita': 'Palpitation',
    'AF': 'Atrial Fibrillation',
    'Atrial Fib': 'Atrial Fibrillation',
    'Fibri': 'Atrial Fibrillation',
    'ALO': 'Acute Lung Oedema',
    'ALI': 'Acute Limb Ischemia',
    'asd': 'Atrial Septal Defect',
    'tb': 'Tuberculosis',
    'ves': 'Ventricular Extrasystole',
    'Hipertensi': 'Hipertensi',
    'pertensi':'Hipertensi',
    'Hipert': 'Hipertensi',
    'Hipere': 'Hipertensi',
    'Hiperl': 'Hipertensi',
    'Hipertiroid': 'Hipertiroid',
    'HFrEF': 'Heart Failure with Reduced Ejection Fraction',
    'OMI': 'Oklusi Miokard Infark',
    'pneumo': 'Pneumonia',
    'cardi': 'Takikardia',
    'kardi': 'Takikardia',
    'asma': 'Asma',
    'anemia': 'Anemia',
    'aortic regur': 'Aortic Regurgitation',
    'aorta regur': 'Aortic Regurgitation',
    'mitral regur': 'Mitral Regurgitation',
    'mr': 'Mitral Regurgitation',
    'rhd': 'Rheumatic Heart Disease',
    'mitral stenosis': 'Mitral Stenosis',
    'mtiral stenosis': 'Mitral Stenosis',
    'ms': 'Mitral Stenosis',
    'aortic stenosis': 'Aortic Stenosis',
    'aorta stenosis': 'Aortic Stenosis',
    'pjb': 'Penyakit Jantung Bawaan',
    'mvp': 'Mitral Valve Prolapse',
    'ihd': 'Ischemic Heart Disease',
    'bacterial infection': 'Bacterial Infection',
    'CA Mam': 'Carcinoma Mammae',
    'CAMam': 'Carcinoma Mammae',
    'Stroke': 'Stroke',
    'snh':'Stroke',
    'PPCM': 'Cardiomiopathy',
    'dyspne': 'Dyspnea',
    'dysn': 'Dyspnea',
    'dyspenu': 'Dyspnea',
    'av blo': 'Atrioventricular Block',
    'valvular heart disease':'Valvular Heart Disease',
    'VSD':'Ventricular Septal Defect',
    'dyspepsia':'Dyspepsia',
    'dysopepsia':'Dyspepsia',
    'HHD':'Hypertensive Heart Disease',
    'malnutrisi':'Malnutrisi',
    'dispep': 'Dyspepsia',
    'Ca prostat': 'Carcinoma Prostat',
    'Ca colon': 'Carcinoma Colon',
    'Aorta dila': 'Aorta Dilatation',
    'Aritmia': 'Aritmia',
    'Limfoma':'Limfoma',
    'AR mild':'Aortic Regurgitation',
    'AR mode':'Aortic Regurgitation',
    'AR severe':'Aortic Regurgitation',
    'Aneurism':'Aunerisma Aorta',
    'AMI':'Infark Miokard Akut',
    'AR s/p':'Aortic Regurgitation',
    'AVA Repair': 'Aortic Valve Area Repair',
    'AVR':'Aortic Valve Area Repair',
    'Acute lung edema': 'Acute Lung Oedema',
    'Edema paru akut':'Acute Lung Oedema',
    'Edem pulmo':'Acute Lung Oedema',
    'Aortis Stenosis':'Aortic Stenosis',
    'Arithmia':'Aritmia',
    'at perbaikan': 'takikardia',
    'Vhest Pain':'Chest Pain',
    'Biliar': 'Atrial Fibrillation',
    'cvi': 'Chronic Venous Insufficiency',
    'dyspep':'Dyspepsia',
    'Dyspesia':'Dyspepsia',
    'dvt': 'Deep Vein Thrombosis',
    'DCM': 'Cardiomiopathy',
    'ht':'Hipertensi',
    'Hiperftensi': 'Hipertensi',
    'Hipetensi': 'Hipertensi',
    'Hipoertensi':'Hipertensi',
    'Ischemic':'Ischemic Heart Disease',
    'cmp': 'Cardiomiopathy',
    'pad':'Peripheral Artery Disease',
    'pda':'Peripheral Artery Disease',
    'pci': 'Percutaneous Coronary Intervention',
    'ph':'Hipertensi',
    'penumonia': 'Pneumonia',
    'ppok':'Penyakit Paru Obstruktif Kronis',
    'ppoki':'Penyakit Paru Obstruktif Kronis',
    'svt':'Takikardia',
    'cap': 'Pneumonia',
    'cpc': 'Cor Pulmonale Chronic',
    'dorv':'Double Outlet Right Ventricle',
    'GERD':'Gastroesophageal Reflux Disease',
    'GEA':'Gastroenteritis Akut',
    'HOCM': 'Cardiomyopathy',
    'LHF':'Left Heart Failure',
    'TOF':'Tetralogy of Fallot',
    'ARDS':'Acute Respiratory Distress Syndrome',
}

for k,v in map1.items():
    replace_func(k,v)

In [ ]:
def replace_func2(torep, repwith):
    df2.loc[df2['diagnosaDokter[0].keterangan'].str.lower() == torep, 'grouped_diagnosa'] = repwith

m2 = {
    'ap': 'Angina Pectoris',
    'ar':'Aortic Regurgitation',
    'as':'Aortic Stenosis',
    'pr':'Pulmonary Regurgitation',
}

for k,v in m2.items():
    replace_func2(k,v)

In [ ]:
df2 = df2.rename(columns={'diagnosaDokter[0].diagnosaa.label' : 'icd10_label'})

In [ ]:
df2_export = df2[['O', 'S', 'P', 'grouped_diagnosa', 'diagnosaDokter[0].keterangan', 'icd10_label', 'pasien.jeniskelamin','pasien.umur','registrasi.kelompokpasien','registrasi.namakelas']]
df2_export.to_csv('cppt_ranap_extract.csv')